# Basal Melt GeoZarr Conversion

In order to display zarr files you hvae to manually build the pyramids similar to a cog file. You also have to link all of the data together, so that I viauliser keeps track of the projections and crs metadata. 
A common specification to work on this is GeoZarr - https://geozarr.org/ .  Which is a composition of other existing conventions.  To create a valid geozarr file we will follow the example of eopf-zarr. To convert a zarr file to a geozarr you have to :
- Converert the dataset zarr into a tree, that represents the data at different scales
- Specify how the groups link to each other
- Specify the projection and transformation information for every grouop

We will follow the eopf-geozarr tree structure so that we can use the new open layers visualisations. This structure has a nested data tree structure where a dataset such as `measurement/reflectance` is a group in the tree has spatial, multiscale and projection infomation. Then the different subgroups are scaled versions of an original dataset, and each subgroup has only spatial information. The highest resolution subgroup is basically the same as teh orignal zarr file.
We will use the basal_melt zarr file as an example.


In [1]:
import xarray as xr
xr.open_datatree('https://s3.explorer.eopf.copernicus.eu/esa-zarr-sentinel-explorer-fra/tests-output/sentinel-2-l2a/S2B_MSIL2A_20260120T125339_N0511_R138_T27VWL_20260120T131151.zarr')

<xarray.DataTree>
Group: /
├── Group: /conditions
│   ├── Group: /conditions/geometry
│   │       Dimensions:                        (angle: 2, band: 13, y: 23, x: 23,
│   │                                           detector: 7)
│   │       Coordinates:
│   │         * angle                          (angle) <U7 56B 'zenith' 'azimuth'
│   │         * band                           (band) <U3 156B 'b00' 'b01' ... 'b11' 'b12'
│   │         * y                              (y) float32 92B 7.1e+06 ... 6.99e+06
│   │         * x                              (x) float32 92B 5e+05 5.05e+05 ... 6.1e+05
│   │         * detector                       (detector) <U3 84B 'd01' 'd02' ... 'd07'
│   │       Data variables:
│   │           mean_sun_angles                (angle) float64 16B ...
│   │           mean_viewing_incidence_angles  (band, angle) float64 208B ...
│   │           sun_angles                     (angle, y, x) float64 8kB ...
│   │           viewing_incidence_angles       (band, detector, angle, y, x) float64 770kB ...
│   ├── Group: /conditions/mask
│   │   ├── Group: /conditions/mask/detector_footprint
│   │   │   ├── Group: /conditions/mask/detector_footprint/r10m
│   │   │   │       Dimensions:  (y: 10980, x: 10980)
│   │   │   │       Coordinates:
│   │   │   │         * y        (y) float32 44kB 7.1e+06 7.1e+06 7.1e+06 ... 6.99e+06 6.99e+06
│   │   │   │         * x        (x) float32 44kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
│   │   │   │       Data variables:
│   │   │   │           b02      (y, x) uint8 121MB ...
│   │   │   │           b03      (y, x) uint8 121MB ...
│   │   │   │           b04      (y, x) uint8 121MB ...
│   │   │   │           b08      (y, x) uint8 121MB ...
│   │   │   ├── Group: /conditions/mask/detector_footprint/r20m
│   │   │   │       Dimensions:  (y: 5490, x: 5490)
│   │   │   │       Coordinates:
│   │   │   │         * y        (y) float32 22kB 7.1e+06 7.1e+06 7.1e+06 ... 6.99e+06 6.99e+06
│   │   │   │         * x        (x) float32 22kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
│   │   │   │       Data variables:
│   │   │   │           b05      (y, x) uint8 30MB ...
│   │   │   │           b06      (y, x) uint8 30MB ...
│   │   │   │           b07      (y, x) uint8 30MB ...
│   │   │   │           b11      (y, x) uint8 30MB ...
│   │   │   │           b12      (y, x) uint8 30MB ...
│   │   │   │           b8a      (y, x) uint8 30MB ...
│   │   │   └── Group: /conditions/mask/detector_footprint/r60m
│   │   │           Dimensions:  (y: 1830, x: 1830)
│   │   │           Coordinates:
│   │   │             * y        (y) float32 7kB 7.1e+06 7.1e+06 7.1e+06 ... 6.99e+06 6.99e+06
│   │   │             * x        (x) float32 7kB 5e+05 5.001e+05 5.001e+05 ... 6.097e+05 6.098e+05
│   │   │           Data variables:
│   │   │               b01      (y, x) uint8 3MB ...
│   │   │               b09      (y, x) uint8 3MB ...
│   │   │               b10      (y, x) uint8 3MB ...
│   │   ├── Group: /conditions/mask/l1c_classification
│   │   │   └── Group: /conditions/mask/l1c_classification/r60m
│   │   │           Dimensions:  (y: 1830, x: 1830)
│   │   │           Coordinates:
│   │   │             * y        (y) float32 7kB 7.1e+06 7.1e+06 7.1e+06 ... 6.99e+06 6.99e+06
│   │   │             * x        (x) float32 7kB 5e+05 5.001e+05 5.001e+05 ... 6.097e+05 6.098e+05
│   │   │           Data variables:
│   │   │               b00      (y, x) uint8 3MB ...
│   │   └── Group: /conditions/mask/l2a_classification
│   │       ├── Group: /conditions/mask/l2a_classification/r20m
│   │       │       Dimensions:  (y: 5490, x: 5490)
│   │       │       Coordinates:
│   │       │         * y        (y) float32 22kB 7.1e+06 7.1e+06 7.1e+06 ... 6.99e+06 6.99e+06
│   │       │         * x        (x) float32 22kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
│   │       │       Data variables:
│   │       │           scl      (y, x) float64 241MB ...
│   │       └── Group: /conditions/mask/l2a_classification

This example uses the basal melt zarr created in the other tutorials,  all source variables use the `x`/`y` spatial dimensions, EPSG:3031, metre coordinates, 512 pixel chunks, and pyramid levels `[1, 2, 4, 8, 16]`.

The output is an EOPF-style GeoZarr hierarchy :

```text
/dfco2
/dfco2/0
/dfco2/1
/dfco2/2
/dfco2/3
/dfco2/4
```

In [3]:
from pathlib import Path
import shutil

from IPython.display import display
import numpy as np
import rioxarray  # activates the .rio accessor used by xarray objects
import zarr

from geozarr_toolkit import (
    MultiscalesConventionMetadata,
    ProjConventionMetadata,
    SpatialConventionMetadata,
    create_proj_attrs,
    create_spatial_attrs,
    create_zarr_conventions,
)


In [4]:
SOURCE_ZARR = 'https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/ocean-soda/dfco2.zarr'
OUTPUT_ZARR =  "oceansoda_dfco2_geozarr.zarr"

NAME = "dfco2"
CRS = "EPSG:4326"
UNITS = "uatm"
CHUNK = 512
PYRAMID_FACTORS = [1, 2, 4, 8]

# overwrite np.nan default nodata value for openlayers visualisaiton
NODATA = -9999.0

In [ ]:
# # save the a subset of the source data locally for faster re-runs

# source = xr.open_zarr(SOURCE_ZARR)
# # limit to med extent
# source = source.sel(lon=slice(-6.062, 36.06), lat=slice(30.27, 45.98))
# # rechunk
# source = source.chunk(chunks={"lat": CHUNK, "lon": CHUNK})


# # clear encoding
# source.dfco2.encoding = {}

# source.to_zarr('dfco2.zarr', zarr_format=3, consolidated=True, align_chunks=True)

## Open the source Zarr

The input store already has `x`, `y`, one or more data variables, and `spatial_ref`. We open it lazily with xarray, attach the known CRS at dataset level, find every variable that uses the spatial dimensions, and convert the encoded nodata value to `NaN` before downsampling. That way `coarsen(...).mean(skipna=True)` ignores nodata pixels for every spatial variable.


In [32]:
SOURCE_ZARR = 'dfco2.zarr'
source = xr.open_zarr(SOURCE_ZARR)

source = source.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs(CRS)

spatial_data_vars = [
    name
    for name, data_array in source.data_vars.items()
    if any(dim in data_array.dims for dim in ("lat", "lon"))
]

non_numeric_spatial_vars = [
    name
    for name in spatial_data_vars
    if not np.issubdtype(source[name].dtype, np.number)
]


for var_name in spatial_data_vars:
    attrs = source[var_name].attrs.copy()
    source[var_name] = source[var_name].where(source[var_name] != NODATA).astype("float32")
    source[var_name].attrs.update(attrs)
    source[var_name].attrs["grid_mapping"] = "spatial_ref"
    if var_name == NAME:
        source[var_name].attrs["units"] = UNITS

In [33]:
{
    "source": str(SOURCE_ZARR),
    "data_variables": list(source.data_vars),
    "spatial_data_variables": spatial_data_vars,
    "shape": [int(source.sizes["lon"]), int(source.sizes["lat"])],
    "crs": CRS,
    "bounds": [float(value) for value in source.rio.bounds()],
    "resolution": [float(value) for value in source.rio.resolution()],
    "nodata_written": float(NODATA),
}

{'source': 'dfco2.zarr',
 'data_variables': ['dfco2'],
 'spatial_data_variables': ['dfco2'],
 'shape': [168, 63],
 'crs': 'EPSG:4326',
 'bounds': [-6.0, 30.25, 36.0, 46.0],
 'resolution': [0.25, 0.25],
 'nodata_written': -9999.0}

## Build the pyramid levels

The native dataset is level `0`. Each following level halves the previous level in both spatial dimensions. Using `Dataset.coarsen(...).mean(skipna=True)` averages all spatial variables together and keeps their coordinate arrays aligned.


In [34]:
if PYRAMID_FACTORS[0] != 1:
    raise ValueError("The first pyramid factor must be 1 for the native-resolution level.")

levels = []
level = source
previous_factor = PYRAMID_FACTORS[0]

for level_index, factor in enumerate(PYRAMID_FACTORS):
    if level_index > 0:
        step = factor / previous_factor
        if not float(step).is_integer():
            raise ValueError(f"Pyramid factor {factor} is not an integer step from {previous_factor}.")

        level = level.coarsen(lon=int(step), lat=int(step), boundary="trim").mean(skipna=True)
        for var_name in spatial_data_vars:
            attrs = source[var_name].attrs.copy()
            level[var_name] = level[var_name].astype("float32")
            level[var_name].attrs.update(attrs)
        level = level.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs(CRS)

    levels.append((factor, level))
    previous_factor = factor

[
    {
        "level": level_index,
        "factor": factor,
        "variables": list(level_ds.data_vars),
        "shape": [int(level_ds.sizes["lon"]), int(level_ds.sizes["lat"])],
        "bounds": [float(value) for value in level_ds.rio.bounds()],
        "resolution": [float(value) for value in level_ds.rio.resolution()],
    }
    for level_index, (factor, level_ds) in enumerate(levels)
]


[{'level': 0,
  'factor': 1,
  'variables': ['dfco2'],
  'shape': [168, 63],
  'bounds': [-6.0, 30.25, 36.0, 46.0],
  'resolution': [0.25, 0.25]},
 {'level': 1,
  'factor': 2,
  'variables': ['dfco2'],
  'shape': [84, 31],
  'bounds': [-6.0, 30.25, 36.0, 45.75],
  'resolution': [0.5, 0.5]},
 {'level': 2,
  'factor': 4,
  'variables': ['dfco2'],
  'shape': [42, 15],
  'bounds': [-6.0, 30.25, 36.0, 45.25],
  'resolution': [1.0, 1.0]},
 {'level': 3,
  'factor': 8,
  'variables': ['dfco2'],
  'shape': [21, 7],
  'bounds': [-6.0, 30.25, 36.0, 44.25],
  'resolution': [2.0, 2.0]}]

## Write the GeoZarr hierarchy

Each pyramid level is written as its own Zarr group under `/basal_melt`. The whole xarray Dataset is written at each level, so all source variables travel together with `x`, `y`, and `spatial_ref` metadata.

Most of the spatial metadata comes directly from rioxarray: `rio.transform()`, `rio.bounds()`, and the xarray dimension sizes.


In [35]:
# Start with a clean output store and parent group.
root = zarr.group(store=OUTPUT_ZARR, zarr_format=3)
root.require_group(NAME)

# Reuse the same compression and CRS metadata at every pyramid level.
compressor = zarr.codecs.ZstdCodec(level=3)
proj_attrs = create_proj_attrs(code=CRS)

layout = []
written_levels = []

for level_index, (factor, level_ds) in enumerate(levels):

    level_name = str(level_index)
    group_path = f"{NAME}/{level_name}"
    transform = level_ds.rio.transform()
    spatial_transform = [
        float(transform.a),
        float(transform.b),
        float(transform.c),
        float(transform.d),
        float(transform.e),
        float(transform.f),
    ]
    spatial_shape = [int(level_ds.sizes["lat"]), int(level_ds.sizes["lon"])]
    spatial_bbox = [float(value) for value in level_ds.rio.bounds()]

    # Attach GeoZarr spatial/proj attrs to this resolution group.
    level_attrs = create_spatial_attrs(
        ["lat", "lon"],
        transform=spatial_transform,
        bbox=spatial_bbox,
        shape=spatial_shape,
        registration="pixel",
    )
    level_attrs.update(proj_attrs)
    level_attrs["grid_mapping"] = "spatial_ref"
    level_attrs["zarr_conventions"] = create_zarr_conventions(
        SpatialConventionMetadata(),
        ProjConventionMetadata(),
    )

    # Fill nodata and keep CF grid_mapping metadata on every spatial variable.
    level_to_write = level_ds.copy()
    for var_name in spatial_data_vars:
        if var_name not in level_to_write:
            continue
        attrs = level_to_write[var_name].attrs.copy()
        level_to_write[var_name] = level_to_write[var_name].fillna(NODATA).astype("float32")
        level_to_write[var_name].attrs.update(attrs)
        level_to_write[var_name].attrs.update(
            {
                "grid_mapping": "spatial_ref",
                "_FillValue": float(NODATA),
                "missing_value": float(NODATA),
            }
        )
        if var_name == NAME:
            level_to_write[var_name].attrs["units"] = UNITS

    # write spatial_ref as a scalar variable in every group
    level_to_write = level_to_write.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs(
        CRS,
        grid_mapping_name="spatial_ref",
    )
    if "spatial_ref" in level_to_write.coords:
        level_to_write = level_to_write.reset_coords("spatial_ref")

    level_to_write.attrs.update(level_attrs)

    # Chunk by variable shape and clear source-store encoding leftovers.
    # In a larger dataset the chunnks should be different for each aggregate level to ensure optimal access
    y_chunk = min(CHUNK, spatial_shape[0])
    x_chunk = min(CHUNK, spatial_shape[1])
    level_to_write = level_to_write.chunk({"lat": y_chunk, "lon": x_chunk})
    for var_name in level_to_write.variables:
        level_to_write[var_name].encoding.clear()

    # Compress and chunk every data variable and coordinate that will be written.
    encoding = {}
    for var_name, variable in level_to_write.variables.items():
        var_encoding = {"compressors": [compressor]}
        if variable.dims:
            var_encoding["chunks"] = tuple(
                y_chunk
                if dim == "lon"
                else x_chunk
                if dim == "lat"
                else min(CHUNK, int(level_to_write.sizes[dim]))
                for dim in variable.dims
            )
        encoding[var_name] = var_encoding

    # Write the dataset for this pyramid level.
    level_to_write.to_zarr(
        OUTPUT_ZARR,
        group=group_path,
        mode="w",
        zarr_format=3,
        compute=True,
        consolidated=False,
        encoding=encoding,
        align_chunks=True
    )

    # Re-apply group attrs after writing so GeoZarr metadata is explicit.
    zarr.open_group(store=OUTPUT_ZARR, path=group_path, mode="a").attrs.update(level_attrs)

    # Record this level for the parent multiscales layout.
    layout_item = {
        "asset": level_name,
        "spatial:shape": spatial_shape,
        "spatial:transform": spatial_transform,
    }
    if level_index > 0:
        scale = float(factor / PYRAMID_FACTORS[level_index - 1])
        layout_item["derived_from"] = str(level_index - 1)
        layout_item["transform"] = {"scale": [scale, scale], "translation": [0.0, 0.0]}

    layout.append(layout_item)
    written_levels.append(
        {
            "path": group_path,
            "factor": factor,
            "variables": list(level_to_write.data_vars),
            "shape": spatial_shape,
            "transform": spatial_transform,
            "bbox": spatial_bbox,
        }
    )

written_levels

[{'path': 'dfco2/0',
  'factor': 1,
  'variables': ['dfco2', 'spatial_ref'],
  'shape': [63, 168],
  'transform': [0.25, 0.0, -6.0, 0.0, 0.25, 30.25],
  'bbox': [-6.0, 30.25, 36.0, 46.0]},
 {'path': 'dfco2/1',
  'factor': 2,
  'variables': ['dfco2', 'spatial_ref'],
  'shape': [31, 84],
  'transform': [0.5, 0.0, -6.0, 0.0, 0.5, 30.25],
  'bbox': [-6.0, 30.25, 36.0, 45.75]},
 {'path': 'dfco2/2',
  'factor': 4,
  'variables': ['dfco2', 'spatial_ref'],
  'shape': [15, 42],
  'transform': [1.0, 0.0, -6.0, 0.0, 1.0, 30.25],
  'bbox': [-6.0, 30.25, 36.0, 45.25]},
 {'path': 'dfco2/3',
  'factor': 8,
  'variables': ['dfco2', 'spatial_ref'],
  'shape': [7, 21],
  'transform': [2.0, 0.0, -6.0, 0.0, 2.0, 30.25],
  'bbox': [-6.0, 30.25, 36.0, 44.25]}]

## Add parent multiscale metadata

The parent `/dfco2` group links the resolution groups together. It stores the GeoZarr conventions, the multiscale layout, the source file path, the source data variable names, and the CRS/spatial metadata shared by the pyramid.


In [36]:
parent_attrs = {
    "zarr_conventions": create_zarr_conventions(
        MultiscalesConventionMetadata(),
        SpatialConventionMetadata(),
        ProjConventionMetadata(),
    ),
    "multiscales": {
        "layout": layout,
        "resampling_method": "average",
    },
    "title": f"{NAME} GeoZarr pyramid",
    "source_file": str(SOURCE_ZARR),
    "data_variables": spatial_data_vars,
}
parent_attrs.update(
    create_spatial_attrs(
        ["lat", "lon"],
        bbox=[float(value) for value in source.rio.bounds()],
        registration="pixel",
    )
)
parent_attrs.update(proj_attrs)

zarr.open_group(store=OUTPUT_ZARR, path=NAME, mode="a").attrs.update(parent_attrs)

parent_attrs


{'zarr_conventions': [{'uuid': 'd35379db-88df-4056-af3a-620245f8e347',
   'schema_url': 'https://raw.githubusercontent.com/zarr-conventions/multiscales/refs/tags/v1/schema.json',
   'spec_url': 'https://github.com/zarr-conventions/multiscales/blob/v1/README.md',
   'name': 'multiscales',
   'description': 'Multiscale layout of zarr datasets'},
  {'uuid': '689b58e2-cf7b-45e0-9fff-9cfc0883d6b4',
   'schema_url': 'https://raw.githubusercontent.com/zarr-conventions/spatial/refs/tags/v1/schema.json',
   'spec_url': 'https://github.com/zarr-conventions/spatial/blob/v1/README.md',
   'name': 'spatial:',
   'description': 'Spatial coordinate information'},
  {'uuid': 'f17cb550-5864-4468-aeb7-f3180cfb622f',
   'schema_url': 'https://raw.githubusercontent.com/zarr-experimental/geo-proj/refs/tags/v1/schema.json',
   'spec_url': 'https://github.com/zarr-experimental/geo-proj/blob/v1/README.md',
   'name': 'proj:',
   'description': 'Coordinate reference system information for geospatial data'}],
 

## Validate using the geozarr_toolkit

In [37]:
from geozarr_toolkit import detect_conventions, validate_group
from zarr.storage import LocalStore

zarr_store = LocalStore(OUTPUT_ZARR)
root = zarr.open_group(zarr_store, mode="r")
pyramid = zarr.open_group(store=zarr_store, path=NAME, mode="r")

print(f"GeoZarr multiscale group: /{NAME}")
print(f"Detected conventions: {detect_conventions(dict(pyramid.attrs))}")

print("\nParent multiscale group validation:")
for convention, errors in validate_group(pyramid).items():
    status = "PASS" if not errors else "FAIL"
    print(f"  [{status}] {convention}")
    for error in errors:
        print(f"         {error}")

print("\nResolution group validation:")
for item in written_levels:
    group = zarr.open_group(store=zarr_store, path=item["path"], mode="r")
    print(f"\n/{item['path']}")
    print(f"  detected: {detect_conventions(dict(group.attrs))}")
    for convention, errors in validate_group(group).items():
        status = "PASS" if not errors else "FAIL"
        print(f"  [{status}] {convention}")
        for error in errors:
            print(f"         {error}")

GeoZarr multiscale group: /dfco2
Detected conventions: ['spatial', 'proj', 'multiscales']

Parent multiscale group validation:
  [PASS] spatial
  [PASS] proj
  [PASS] multiscales
  [PASS] zarr_conventions

Resolution group validation:

/dfco2/0
  detected: ['spatial', 'proj']
  [PASS] spatial
  [PASS] proj
  [PASS] zarr_conventions

/dfco2/1
  detected: ['spatial', 'proj']
  [PASS] spatial
  [PASS] proj
  [PASS] zarr_conventions

/dfco2/2
  detected: ['spatial', 'proj']
  [PASS] spatial
  [PASS] proj
  [PASS] zarr_conventions

/dfco2/3
  detected: ['spatial', 'proj']
  [PASS] spatial
  [PASS] proj
  [PASS] zarr_conventions


In [52]:
import zarr

zarr.consolidate_metadata(
    store=OUTPUT_ZARR,
)

/home/krasen/hackathon_websites/ocean_hackathon/.pixi/envs/default/lib/python3.13/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


<Group file://oceansoda_dfco2_geozarr.zarr>

In [53]:
display(xr.open_datatree(OUTPUT_ZARR))

<xarray.DataTree>
Group: /
└── Group: /dfco2
    │   Attributes:
    │       zarr_conventions:        [{'uuid': 'd35379db-88df-4056-af3a-620245f8e347'...
    │       multiscales:             {'layout': [{'asset': '0', 'spatial:shape': [63,...
    │       title:                   dfco2 GeoZarr pyramid
    │       source_file:             dfco2.zarr
    │       data_variables:          ['dfco2']
    │       spatial:dimensions:      ['lat', 'lon']
    │       spatial:bbox:            [-6.0, 30.25, 36.0, 46.0]
    │       spatial:transform_type:  affine
    │       spatial:registration:    pixel
    │       proj:code:               EPSG:4326
    ├── Group: /dfco2/0
    │       Dimensions:      (time: 1886, lat: 63, lon: 168)
    │       Coordinates:
    │         * time         (time) datetime64[ns] 15kB 1982-01-04 1982-01-12 ... 2022-12-30
    │         * lat          (lat) float64 504B 30.38 30.62 30.88 ... 45.38 45.62 45.88
    │         * lon          (lon) float64 1kB -5.875 -5.625 -5.375 ... 35.38 35.62 35.88
    │       Data variables:
    │           dfco2        (time, lat, lon) float32 80MB ...
    │           spatial_ref  int64 8B ...
    │       Attributes: (12/23)
    │           contact:                   gregorl@ethz.ch
    │           affiliation:               ETH Zuerich
    │           product_name:              OceanSODA-ETHZ-HR
    │           product_version:           v2023.01
    │           project:                   ESA-OceanHealth-OceanAcidification
    │           product_details:           Fugacity of CO2 (fCO2) and total alkalinity ar...
    │           ...                        ...
    │           spatial:transform:         [0.25, 0.0, -6.0, 0.0, 0.25, 30.25]
    │           spatial:shape:             [63, 168]
    │           spatial:registration:      pixel
    │           proj:code:                 EPSG:4326
    │           grid_mapping:              spatial_ref
    │           zarr_conventions:          [{'uuid': '689b58e2-cf7b-45e0-9fff-9cfc0883d6b...
    ├── Group: /dfco2/1
    │       Dimensions:      (time: 1886, lat: 31, lon: 84)
    │       Coordinates:
    │         * time         (time) datetime64[ns] 15kB 1982-01-04 1982-01-12 ... 2022-12-30
    │         * lat          (lat) float64 248B 30.5 31.0 31.5 32.0 ... 44.0 44.5 45.0 45.5
    │         * lon          (lon) float64 672B -5.75 -5.25 -4.75 ... 34.75 35.25 35.75
    │       Data variables:
    │           dfco2        (time, lat, lon) float32 20MB ...
    │           spatial_ref  int64 8B ...
    │       Attributes: (12/23)
    │           contact:                   gregorl@ethz.ch
    │           affiliation:               ETH Zuerich
    │           product_name:              OceanSODA-ETHZ-HR
    │           product_version:           v2023.01
    │           project:                   ESA-OceanHealth-OceanAcidification
    │           product_details:           Fugacity of CO2 (fCO2) and total alkalinity ar...
    │           ...                        ...
    │           spatial:transform:         [0.5, 0.0, -6.0, 0.0, 0.5, 30.25]
    │           spatial:shape:             [31, 84]
    │           spatial:registration:      pixel
    │           proj:code:                 EPSG:4326
    │           grid_mapping:              spatial_ref
    │           zarr_conventions:          [{'uuid': '689b58e2-cf7b-45e0-9fff-9cfc0883d6b...
    ├── Group: /dfco2/2
    │       Dimensions:      (time: 1886, lat: 15, lon: 42)
    │       Coordinates:
    │         * time         (time) datetime64[ns] 15kB 1982-01-04 1982-01-12 ... 2022-12-30
    │         * lat          (lat) float64 120B 30.75 31.75 32.75 ... 42.75 43.75 44.75
    │         * lon          (lon) float64 336B -5.5 -4.5 -3.5 -2.5 ... 32.5 33.5 34.5 35.5
    │       Data variables:
    │           dfco2        (time, lat, lon) float32 5MB ...
    │           spatial_ref  int64 8B ...
    │       Attributes: (12/23)
    │           contact:                   gregorl@ethz.ch
    │

## Visualising GeoZarr data

To actually visualise the data you need a viewer that reads geozarr datasets such as open payers 10.8 or above. And you need to serve the files through http or s3. So, youll need:

#### 1. Upload the data to an s3 server, or simulate an http server locally using python.


#### 2. Create an html file that creates the open layers Geozar layer viewer that points to the file. 

You can see the example visualiser for the javascript code needed: https://sunnydean.github.io/ocean_cubes_vis/ .